# Smart Batching Search - Testing Notebook

This notebook uses **[bigdata-smart-batching](https://pypi.org/project/bigdata-smart-batching/)** on PyPI: semantic search with intelligent company grouping, proportional sampling, and rate-limited parallel execution.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [1]:
# Library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

# Set API base URL BEFORE importing (smart_batching_config reads it at import time)
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

# Import utilities from bigdata-smart-batching package
from bigdata_smart_batching import (
    plan_search,
    execute_search,
    deduplicate_documents,
    save_plan,
    load_plan,
    load_universe_from_csv,
    convert_to_dataframe,
)

## 2. Configuration

In [2]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
TEST_TEXT = "The company has been impacted by strait of hormuz shipping disruption"
TEST_UNIVERSE_CSV = "id_name_mapping_us_top_3000.csv"
# TEST_UNIVERSE_CSV = "sample_universe.csv"  # small test universe
TEST_START_DATE = "2026-03-01"
TEST_END_DATE = "2026-06-01"
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV}")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v2_pz...pj04

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'The company has been impacted by strait of hormuz shipping disruption'
   Universe: id_name_mapping_us_top_3000.csv
   Date Range: 2026-03-01 to 2026-06-01
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [3]:
# Test loading universe from CSV
try:
    companies = load_universe_from_csv(TEST_UNIVERSE_CSV)
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

2026-06-09 17:36:23,685 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
✅ Loaded 4731 companies from id_name_mapping_us_top_3000.csv
   First 5 companies: ['00067A', '001F1B', '002A99', '00326D', '003B70']


## 4. Step 1: Plan Search

In [53]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)
    
    try:
        plan = plan_search(
            text=TEST_TEXT,
            universe=TEST_UNIVERSE_CSV,
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
            volume_query_mode="iterative",
            max_iterations_per_batch=10
        )
        
        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['chunk_upper_bound_estimate']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")
        
        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")
        
        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")
        
    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-06-02 16:38:35,590 - INFO - Planning search for text: 'The company has been impacted by strait of hormuz shipping disruption'
2026-06-02 16:38:35,593 - INFO - Date range: 2026-03-01 to 2026-06-01
2026-06-02 16:38:35,615 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
2026-06-02 16:38:35,616 - INFO - Loaded 4731 companies from universe
2026-06-02 16:38:35,633 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting
2026-06-02 16:38:35,675 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
PHASE 1: Querying full period for all companies (2026-03-01 to 2026-06-01)
         Mode: iterative
    [ITERATIVE MODE] Querying 4731 companies in 10 batches of 500 across 1 date sub-range(s) (10 work items, max_workers=8)
    Each batch iterates until no new companies are found (max 10 iterations)
      Batch 4/10, Iter 1: Found 219 new compa

## 5. Save Plan (Optional)

In [31]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-06-02 16:13:49,479 - INFO - Plan saved to test_search_plan.json
✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [54]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results_raw = execute_search(
            search_plan=plan,
            chunk_percentage=TEST_CHUNK_PERCENTAGE,
            requests_per_minute=450,  # Rate limit
            basket_filtered_entities=True,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
        )
        
        results = deduplicate_documents(results_raw)

        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} deduplicated chunks")
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-06-02 16:39:11,066 - INFO - Executing search with 10.0% of chunks
2026-06-02 16:39:11,079 - INFO - Total maximum expected chunks: 19,659
2026-06-02 16:39:11,145 - INFO - Searching 201 baskets
2026-06-02 16:39:13,574 - INFO - Basket basket_100_high_20260301_20260601: Retrieved 52 documents with 53 chunks
2026-06-02 16:39:15,909 - INFO - Basket basket_101_high_20260301_20260601: Retrieved 51 documents with 52 chunks
2026-06-02 16:39:16,002 - INFO - Basket basket_102_high_20260301_20260601: Retrieved 51 documents with 51 chunks
2026-06-02 16:39:16,096 - INFO - Basket basket_106_medium_20260301_20260601: Retrieved 74 documents with 91 chunks
2026-06-02 16:39:16,410 - INFO - Basket basket_108_medium_20260301_20260601: Retrieved 53 documents with 85 chunks
2026-06-02 16:39:16,473 - INFO - Basket basket_109_medium_20260301_20260601: Retrieved 79 documents with 82 chunks
2026-06-02 16:39

## 7. Analyze Results

In [55]:
results[2]

{'id': 'B9F3AE43247D11D2B1B30FCBF731BD38',
 'headline': 'Stock Market Today, May 4: Strait of Hormuz Tensions Weigh on Stocks at Midday',
 'timestamp': '2026-05-04T17:45:49',
 'source': {'id': 'E5AA62', 'name': 'Yahoo! Finance', 'rank': 'RANK_2'},
 'url': 'https://finance.yahoo.com/news/stock-market-today-may-4-170625378.html',
 'chunks': [{'cnum': 2,
   'text': "Investors will be watching upcoming earnings from Advanced Micro Devices and Palantir Technologies to assess the strength of artificial intelligence (AI) and broader tech leadership.\nWhat this means for investors\nFears of escalating tensions in the Middle East weighed on markets this morning, causing headline-driven volatility. WTI crude had reached $105 a barrel by midday, close to a four-year high, as a renewed focus on traffic through the Strait of Hormuz pushed up oil prices and pressured stocks.\nIn recent weeks, resilient first-quarter earnings have outweighed energy concerns, particularly in tech, where megacaps have 

In [56]:
# Convert to DataFrame (exploded by chunk)
df = convert_to_dataframe(results)
df.head(20)

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,detections,url,reporting_entities
0,2026-05-04,EEA37AA18E73ABB6F937CCD8376259B6,"Stock Market Today, May 4: Strait of Hormuz Te...",7DFD4A,Nasdaq,RANK_2,2,What this means for investors\nFears of escala...,0.290272,-0.48,[F1C69A],"[{'id': 'C4F920', 'start': 315, 'end': 321, 't...",https://www.nasdaq.com/articles/stock-market-t...,[]
1,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,8,Palantir Technologies Inc. is up 12.84% as def...,0.274425,-0.03,[F1C69A],"[{'id': '913660', 'start': 286, 'end': 294, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
2,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,5,Return Since Feb. 27 close\n|-\n| Red Cat Hold...,0.054242,0.09,[F1C69A],"[{'id': '5C3ECD', 'start': 495, 'end': 499, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
3,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,12,"Logistics names - United Parcel Service Inc., ...",0.276596,-0.40,[FAE021],"[{'id': '555EB5', 'start': 162, 'end': 169, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
4,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,9,Return\n|-\n| Alaska Air Group Inc. (NYSE: ALK...,0.108367,-0.02,[FAE021],"[{'id': '751A74', 'start': 566, 'end': 607, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
5,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,11,Cruise lines have been hit almost as hard. Car...,0.238561,-0.75,[56CC0A],"[{'id': '067779', 'start': 43, 'end': 57, 'typ...",https://www.benzinga.com/node/51313150?utm_cam...,[]
6,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,7,Refiners like Marathon Petroleum Corp. and Val...,0.054774,0.26,"[641F17, 2D2D43]","[{'id': '42B2B7', 'start': 165, 'end': 173, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
7,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,10,The reopen basket captures names whose earning...,0.126313,-0.64,[CF6A5A],"[{'id': '23E2A4', 'start': 226, 'end': 231, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
8,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,6,Red Cat Holdings leads the basket with a 40.95...,0.099512,0.75,[9C5BA5],"[{'id': '209994', 'start': 228, 'end': 241, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]
9,2026-03-17,666A02E30185B4CC61EABAF77A4F45EE,Markets Fear Prolonged Iran War - These 2 'Hor...,5A5702,Benzinga,RANK_1,18,The bank warned that if the closure extends to...,0.147514,-0.58,[E70531],"[{'id': 'FE1757', 'start': 100, 'end': 101, 't...",https://www.benzinga.com/node/51313150?utm_cam...,[]


In [57]:
top10 = df.sort_values(by="chunk_relevance", ascending=False).head(10)
print(top10)

             date                            doc_id  \
1798   2026-03-02  E5B5670E9FEC3A0E7A53813A45DAFD82   
14954  2026-05-10  46EB82FD7A0112ED5DD4D896737C29F0   
2485   2026-03-03  8AF26D4ACF0E35863868D7E905ECBB87   
9888   2026-03-11  EC9F132C2C4D5EE783053E454DB1C56F   
1254   2026-03-23  899494E0C3F5A6521A6E50CAA0C004CC   
4510   2026-05-13  BA63E3AC50DD1F1E5009957682B20E72   
5907   2026-05-11  0F00A980EC566A9A7821281BC1824069   
6434   2026-05-27  514796A86E3B4FED4489CA5B66F16B4B   
13442  2026-05-22  021DDB7D52562AB46E22B1439F85AFB7   
5022   2026-04-24  E5AFD735407F97A8B64DE552D8747DA2   

                                                headline source_id  \
1798   The $100 Oil Question: What Happens If The Str...    5A5702   
14954  Aramco CEO Warns Of Multi-Year Energy Disrupti...    5A5702   
2485   Market Chatter: 'About 10%' of Global Containe...    8C739A   
9888   Persistent Disruptions From Strait of Hormuz C...    8C739A   
1254   Iranian attacks on Strait of Hormuz a

In [59]:
top10

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,detections,url,reporting_entities
1798,2026-03-02,E5B5670E9FEC3A0E7A53813A45DAFD82,The $100 Oil Question: What Happens If The Str...,5A5702,Benzinga,RANK_1,3,"In a note Monday, Goldman Sachs commodity anal...",0.727226,-0.58,[50070E],"[{'id': 'F7E3B2', 'start': 169, 'end': 177, 't...",https://www.benzinga.com/node/50982507?utm_cam...,[]
14954,2026-05-10,46EB82FD7A0112ED5DD4D896737C29F0,Aramco CEO Warns Of Multi-Year Energy Disrupti...,5A5702,Benzinga,RANK_1,1,Global energy markets will rebalance in a few ...,0.695298,-0.73,[061856],"[{'id': 'F20FAB', 'start': 740, 'end': 752, 't...",https://www.benzinga.com/node/52437708?utm_cam...,[]
2485,2026-03-03,8AF26D4ACF0E35863868D7E905ECBB87,Market Chatter: 'About 10%' of Global Containe...,8C739A,MT Newswires - Asia Pacific,RANK_1,1,"08:54 PM EST, 03/02/2026 (MT Newswires) -- Abo...",0.662911,-0.51,[061856],"[{'id': '061856', 'start': 175, 'end': 182, 't...",,[]
9888,2026-03-11,EC9F132C2C4D5EE783053E454DB1C56F,Persistent Disruptions From Strait of Hormuz C...,8C739A,MT Newswires - Asia Pacific,RANK_1,3,Key factors to assess under a persistent disru...,0.661155,-0.48,[CFF97C],"[{'id': '23E2A4', 'start': 145, 'end': 150, 't...",,[]
1254,2026-03-23,899494E0C3F5A6521A6E50CAA0C004CC,Iranian attacks on Strait of Hormuz an 'act of...,DA9FC6,Financial Times,RANK_1,5,The conflict had forced Adnoc to adjust its ou...,0.660617,-0.77,[940C3D],"[{'id': '277B39', 'start': 439, 'end': 444, 't...",https://www.ft.com/content/1a56c386-d0cc-48f5-...,[]
4510,2026-05-13,BA63E3AC50DD1F1E5009957682B20E72,"TORM: Q1 2026 Earnings Call on May 13, 2026 - ...",DA0F7F,Quartr Transcripts,RANK_1,10,The more important explanation lies on the ton...,0.651926,-0.69,[7697D2],"[{'id': '555EB5', 'start': 553, 'end': 558, 't...",,[]
5907,2026-05-11,0F00A980EC566A9A7821281BC1824069,The Mosaic Company: Q1 2026 Earnings Call on M...,28DED6,Quartr Reports,RANK_1,139,"In addition, geopolitical instability and heig...",0.648250,-0.77,[9C5BA5],"[{'id': '9225D1', 'start': 676, 'end': 686, 't...",https://files.quartr.com/reports/59103-2026-05...,[]
6434,2026-05-27,514796A86E3B4FED4489CA5B66F16B4B,Hafnia Limited: Q1 2026 Earnings Call on May 2...,E38350,Quartr Presentation Materials,RANK_1,57,TRAPPED VESSELS BY CLASS\nLaden Ballast\nThe c...,0.637490,-0.57,[C69A57],"[{'id': 'FD9CFE', 'start': 174, 'end': 179, 't...",https://files.quartr.com/conference-calls/5799...,[]
13442,2026-05-22,021DDB7D52562AB46E22B1439F85AFB7,Legal disputes rip through oil shipping after ...,DA9FC6,Financial Times,RANK_1,1,TotalEnergies is weighing taking legal action ...,0.636948,-0.73,[D01300],"[{'id': '89FCE6', 'start': 512, 'end': 519, 't...",https://www.ft.com/content/d2cb463e-fa6e-47c1-...,[]
5022,2026-04-24,E5AFD735407F97A8B64DE552D8747DA2,Heidmar CEO On Tanker Rates Amid Gulf Disruption,5A5702,Benzinga,RANK_1,1,The oil tanker market is experiencing a rapid ...,0.629999,-0.74,[D82H3G],"[{'id': '1DF8AA', 'start': 155, 'end': 168, 't...",https://www.benzinga.com/node/52040254?utm_cam...,[]


In [58]:
for i, headline in enumerate(top10["chunk_text"], start=1):
    print(f"{i}. {headline}")

1. In a note Monday, Goldman Sachs commodity analyst Daan Struyven said tanker traffic through the Strait appears "significantly disrupted," as shippers, oil producers and insurers adopt a cautious stance amid reports of damaged vessels.
2. Global energy markets will rebalance in a few months only if crude and liquefied natural gas (LNG) shipments through the Strait of Hormuz resume immediately, Aramco CEO Amin Nasser said on Sunday.
Nasser warned that market disruption could extend into 2027 if shipping remains curtailed by more than a few weeks from today, Bloomberg News reported, citing an emailed statement from Aramco. He told Reuters separately that the global oil market lost about 1 billion barrels over the past two months due to shipping disruptions through the strait.
The strait, where about one-fifth of the world's oil and LNG typically flows through the waterway, became a flashpoint after U.S. military operations against Iran started on February 28. Iran's restrictions on tan

In [38]:
source_counts = df.groupby('entity_ids').size().sort_values(ascending=False)

TypeError: unhashable type: 'list'

In [37]:
#Analyze results
if results:

    print("📈 Results Analysis")
    print("-" * 80)
    
    # Summary stats
    n_docs = df['doc_id'].nunique()
    n_chunks = len(df)
    print(f"\n   Total: {n_docs:,} documents, {n_chunks:,} chunks")
    
    # Relevance distribution
    if 'chunk_relevance' in df.columns and df['chunk_relevance'].notna().any():
        print(f"\n   Relevance Scores:")
        print(f"     Min: {df['chunk_relevance'].min():.3f}")
        print(f"     Max: {df['chunk_relevance'].max():.3f}")
        print(f"     Avg: {df['chunk_relevance'].mean():.3f}")
    
    # Sentiment distribution
    if 'chunk_sentiment' in df.columns and df['chunk_sentiment'].notna().any():
        sentiments = df['chunk_sentiment'].dropna()
        positive = (sentiments > 0).sum()
        negative = (sentiments < 0).sum()
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    if 'source_name' in df.columns:
        source_counts = df.groupby('source_name').size().sort_values(ascending=False)
        print(f"\n   Top Sources:")
        for source, count in source_counts.head(5).items():
            print(f"     {source}: {count} chunks")
    
    # Show DataFrame info
    print(f"\n   DataFrame shape: {df.shape}")
    
    # Save results
    from datetime import datetime
    results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    df.to_json(results_file, orient='records', indent=2)
    print(f"\n💾 Saved to {results_file}")

else:
    df = None
    print("⚠️  No results to analyze")

📈 Results Analysis
--------------------------------------------------------------------------------

   Total: 1,716 documents, 1,827 chunks

   Relevance Scores:
     Min: 0.002
     Max: 0.655
     Avg: 0.166

   Sentiment Distribution:
     Positive: 836 (45.8%)
     Negative: 873 (47.8%)
     Neutral: 118 (6.5%)

   Top Sources:
     Benzinga: 257 chunks
     US News & World Report: 164 chunks
     AOL.com: 143 chunks
     Yahoo! Finance: 129 chunks
     The Globe And Mail: 68 chunks

   DataFrame shape: (1827, 14)

💾 Saved to search_results_20260602_162136.json


## 8. Summary

In [9]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['chunk_upper_bound_estimate']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['chunk_upper_bound_estimate']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 73,507
   Baskets created: 81
✅ Execution: SUCCESS
   Chunks retrieved: 5,093
   Percentage used: 10%
   Actual vs Expected: 6.9%

Test complete!


## 9. Load Saved Plan (Optional)

In [10]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('chunk_upper_bound_estimate', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-04-17 15:58:33,058 - INFO - Plan loaded from test_search_plan.json
✅ Plan loaded from test_search_plan.json
   Total expected chunks: 69,426
   Number of baskets: 77

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)
